# 03 - HydroServer ETL Demo

## Setup and Creation Controls
This short notebook shows only the HTTPExtractor path for GEOGLOWS Aroca River forecasts.

In [ ]:
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path

import pandas as pd

try:
    from hydroserverpy import HydroServer
except Exception as exc:
    HydroServer = None
    print(f"hydroserverpy is not available yet: {exc}")

HYDROSERVER_HOST = "https://playground.hydroserver.org"
WORKSPACE_NAME = "hydroserver_uganda_demo"
WORKSPACE_IS_PRIVATE = False

# Choose one: "anonymous" or "api_key".
AUTH_METHOD = "api_key"
HYDROSERVER_API_KEY = ""  # Keep secrets out of saved notebooks. Paste only when prompted.

# Facilitator controls. Defaults keep notebooks safe for anonymous/local runs.
CREATE_WORKSPACE_IF_MISSING = False
DELETE_CREATED_RESOURCES_AT_END = False
DEMO_RESOURCE_PREFIX = "Uganda Demo"
DEMO_RUN_SUFFIX = ""

# Google Colab/local path support. Leave blank unless data is somewhere custom.
DATA_DIR_OVERRIDE = ""


def resolve_data_dir(data_dir_override=""):
    candidates = []
    if data_dir_override:
        candidates.append(Path(data_dir_override).expanduser())
    candidates.extend([
        Path("data"),
        Path("../data"),
        Path("hydroserver_workshop/data"),
        Path("../hydroserver_workshop/data"),
        Path("/content/hydroserver_workshop/data"),
        Path("/content/data"),
    ])
    for candidate in candidates:
        if candidate.exists() and (candidate / "Uganda_Hydroweb.csv").exists():
            return candidate
    searched = "\n".join(f"- {candidate}" for candidate in candidates)
    raise FileNotFoundError(
        "Could not find the workshop data folder. Upload hydroserver_workshop/data, "
        "upload data/ beside the notebook, or set DATA_DIR_OVERRIDE. Searched:\n"
        f"{searched}"
    )


DATA_DIR = resolve_data_dir(DATA_DIR_OVERRIDE)
STATION_CATALOG_CSV = DATA_DIR / "Uganda_Hydroweb.csv"
SELECTED_STATION_CSV = DATA_DIR / "uganda_selected_station.csv"
HYDROWEB_DIR = DATA_DIR / "hydroweb"
GEOGLOWS_DIR = DATA_DIR / "geoglows"
STREAMFLOW_CSV = DATA_DIR / "sample_streamflow_observations.csv"
FORECAST_CSV = DATA_DIR / "sample_forecast_timeseries.csv"

demo_run_suffix = DEMO_RUN_SUFFIX or datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
demo_resource_prefix = f"{DEMO_RESOURCE_PREFIX} {demo_run_suffix}"

print(f"HydroServer host: {HYDROSERVER_HOST}")
print(f"Workspace: {WORKSPACE_NAME}")
print(f"Authentication mode: {AUTH_METHOD}")
print(f"Data directory: {DATA_DIR}")

GEOGLOWS_API_ROOT = "https://geoglows.ecmwf.int/api"
AROCA_GEOGLOWS_RIVER_ID = "160180844"
AROCA_LATITUDE = 1.533355
AROCA_LONGITUDE = 32.21666
RUN_ETL = True  # Keep False until the endpoint and target datastream are verified live.
AROCA_FORECAST_DATASTREAM_ID = ""

HydroServer host: https://playground.hydroserver.org
Workspace: hydroserver_uganda_demo
Authentication mode: api_key
Data directory: ../data


## Connect to HydroServer

In [9]:
def _prompt_if_needed(value, prompt):
    return value if value else getpass(prompt)

hs_api = None

if HydroServer is None:
    print("Install hydroserverpy before connecting to HydroServer.")
elif AUTH_METHOD == "anonymous":
    try:
        hs_api = HydroServer(host=HYDROSERVER_HOST)
        print("Connected anonymously. Anonymous mode can read public data but cannot create or upload resources.")
    except Exception as exc:
        print(f"Anonymous connection failed: {exc}")
elif AUTH_METHOD == "api_key":
    api_key = _prompt_if_needed(HYDROSERVER_API_KEY, "HydroServer API key: ")
    try:
        hs_api = HydroServer(host=HYDROSERVER_HOST, apikey=api_key)
        print("Connected with API-key authentication.")
    except Exception as exc:
        print(f"API-key connection failed: {exc}")
else:
    raise ValueError("AUTH_METHOD must be 'anonymous' or 'api_key'.")

Connected with API-key authentication.


## Track Created Resources

In [10]:
created_resources = []


def resource_uid(resource):
    if resource is None:
        return None
    if isinstance(resource, str):
        return resource
    if isinstance(resource, dict):
        for key in ("uid", "id", "workspace_id"):
            if resource.get(key):
                return str(resource[key])
        return None
    for attr in ("uid", "id", "workspace_id"):
        value = getattr(resource, attr, None)
        if value:
            return str(value)
    if hasattr(resource, "model_dump"):
        dumped = resource.model_dump()
        if isinstance(dumped, dict):
            return resource_uid(dumped)
    return None


def resource_name(resource):
    if resource is None:
        return None
    if isinstance(resource, dict):
        for key in ("name", "code", "definition", "sampling_feature_code"):
            if resource.get(key):
                return str(resource[key])
        return None
    for attr in ("name", "code", "definition", "sampling_feature_code"):
        value = getattr(resource, attr, None)
        if value:
            return str(value)
    if hasattr(resource, "model_dump"):
        dumped = resource.model_dump()
        if isinstance(dumped, dict):
            return resource_name(dumped)
    return type(resource).__name__


def record_resource(resource_type, resource, station_id=None, source=None):
    created_resources.append({
        "resource_type": resource_type,
        "station_id": station_id,
        "source": source,
        "name_or_code": resource_name(resource),
        "uuid": resource_uid(resource),
        "python_type": type(resource).__name__,
        "resource": resource,
    })
    print(f"{resource_type}: {resource_name(resource)} | uuid={resource_uid(resource)}")
    return resource


def created_resources_dataframe(include_objects=False):
    rows = []
    for row in created_resources:
        rows.append({key: value for key, value in row.items() if include_objects or key != "resource"})
    return pd.DataFrame(rows)

print("Resource registry initialized.")

Resource registry initialized.


## Find or Optionally Create Demo Workspace

In [11]:
workspace = None
workspace_uid = None

if hs_api is None:
    print("Skipping workspace lookup because the HydroServer client is unavailable.")
elif AUTH_METHOD == "anonymous":
    print(f"Anonymous mode: cannot create or manage workspace '{WORKSPACE_NAME}'.")
    print("Switch AUTH_METHOD to 'api_key' for live creation/upload demos.")
else:
    try:
        workspaces = hs_api.workspaces.list(fetch_all=True)
        workspace_items = getattr(workspaces, "items", workspaces)
        workspace = next((item for item in workspace_items if getattr(item, "name", None) == WORKSPACE_NAME), None)
        if workspace is None and CREATE_WORKSPACE_IF_MISSING:
            workspace = record_resource(
                "workspace",
                hs_api.workspaces.create(name=WORKSPACE_NAME, is_private=WORKSPACE_IS_PRIVATE),
            )
        elif workspace is None:
            print(f"Workspace '{WORKSPACE_NAME}' was not found. Ask the facilitator to create it first.")
        else:
            print(f"Using existing workspace: {getattr(workspace, 'name', WORKSPACE_NAME)}")
        workspace_uid = resource_uid(workspace)
        if workspace_uid:
            print(f"Workspace UUID: {workspace_uid}")
    except Exception as exc:
        print(f"Could not find or create workspace '{WORKSPACE_NAME}': {exc}")

Using existing workspace: hydroserver_uganda_demo
Workspace UUID: 019dcc03-9020-718b-9c66-d9da3401eede


## Create Needed Aroca Forecast Metadata

In [12]:
aroca_thing_template = {
    "name": f"{demo_resource_prefix} Aroca River GEOGLOWS Forecast",
    "description": "Aroca River forecast point for the GEOGLOWS HTTPExtractor ETL demo.",
    "sampling_feature_type": "Site",
    "sampling_feature_code": f"AROCA_{AROCA_GEOGLOWS_RIVER_ID}",
    "site_type": "Stream",
    "latitude": AROCA_LATITUDE,
    "longitude": AROCA_LONGITUDE,
    "elevation_m": 0.0,
    "elevation_datum": "WGS84",
    "state": "",
    "county": "Aroca River",
    "country": "UG",
    "data_disclaimer": "Workshop forecast demonstration data from the GEOGLOWS REST API.",
    "is_private": False,
    "workspace": workspace_uid or "<workspace-uuid>",
}
forecast_property_template = {
    "name": f"{demo_resource_prefix} Forecast Streamflow",
    "definition": "Forecast water discharge in a river channel",
    "description": "GEOGLOWS forecast streamflow for the Aroca River point.",
    "observed_property_type": "Hydrology",
    "code": f"ForecastStreamflow_{demo_run_suffix}",
    "workspace": workspace_uid or "<workspace-uuid>",
}
forecast_unit_template = {
    "name": f"{demo_resource_prefix} Cubic meters per second",
    "symbol": "m3/s",
    "definition": "Cubic meters per second",
    "unit_type": "Discharge",
    "workspace": workspace_uid or "<workspace-uuid>",
}
forecast_sensor_template = {
    "name": f"{demo_resource_prefix} GEOGLOWS Forecast API",
    "description": "GEOGLOWS REST forecast endpoint.",
    "encoding_type": "application/json",
    "manufacturer": "GEOGLOWS",
    "sensor_model": "GEOGLOWS Forecast",
    "sensor_model_link": GEOGLOWS_API_ROOT,
    "method_type": "Model Forecast",
    "method_link": GEOGLOWS_API_ROOT,
    "method_code": f"GEOGLOWS_FORECAST_{demo_run_suffix}",
    "workspace": workspace_uid or "<workspace-uuid>",
}
forecast_processing_level_template = {
    "code": f"FORECAST_{demo_run_suffix}",
    "definition": "Forecast",
    "explanation": "Forecast values generated by GEOGLOWS.",
    "workspace": workspace_uid or "<workspace-uuid>",
}

def build_forecast_datastream_template(thing, sensor, observed_property, processing_level, unit):
    return {
        "name": f"{demo_resource_prefix} Aroca River GEOGLOWS Forecast",
        "description": "GEOGLOWS forecast streamflow loaded through hydroserverpy.etl.",
        "observation_type": "Model Forecast",
        "sampled_medium": "Water",
        "no_data_value": -9999,
        "aggregation_statistic": "Continuous",
        "time_aggregation_interval": 1,
        "status": "Ongoing",
        "result_type": "Timeseries",
        "value_count": 0,
        "phenomenon_begin_time": datetime.now(timezone.utc),
        "phenomenon_end_time": None,
        "result_begin_time": datetime.now(timezone.utc),
        "result_end_time": None,
        "is_visible": True,
        "is_private": False,
        "thing": resource_uid(thing),
        "sensor": resource_uid(sensor),
        "observed_property": resource_uid(observed_property),
        "processing_level": resource_uid(processing_level),
        "unit": resource_uid(unit),
        "time_aggregation_interval_unit": "hours",
        "intended_time_spacing": 1,
        "intended_time_spacing_unit": "hours",
    }

forecast_datastream = None
if hs_api is None or workspace is None or workspace_uid is None:
    print("Skipping Aroca resource creation because an authenticated workspace is required.")
else:
    try:
        thing = record_resource("thing", hs_api.things.create(**aroca_thing_template), source="Aroca River")
        observed_property = record_resource("observed_property", hs_api.observedproperties.create(**forecast_property_template))
        unit = record_resource("unit", hs_api.units.create(**forecast_unit_template))
        sensor = record_resource("sensor", hs_api.sensors.create(**forecast_sensor_template))
        processing_level = record_resource("processing_level", hs_api.processinglevels.create(**forecast_processing_level_template))
        forecast_datastream = record_resource(
            "datastream",
            hs_api.datastreams.create(**build_forecast_datastream_template(thing, sensor, observed_property, processing_level, unit)),
            source="GEOGLOWS Forecast",
        )
        AROCA_FORECAST_DATASTREAM_ID = resource_uid(forecast_datastream)
    except Exception as exc:
        print(f"Could not create Aroca forecast resources: {exc}")

display(created_resources_dataframe())

thing: Uganda Demo 20260427222424 Aroca River GEOGLOWS Forecast | uuid=019dd10b-68b2-7e63-a68b-851252b222b7
observed_property: Uganda Demo 20260427222424 Forecast Streamflow | uuid=019dd10b-6c18-785f-98f9-fc7073faef17
unit: Uganda Demo 20260427222424 Cubic meters per second | uuid=019dd10b-6f7f-74b8-a8a4-5d7ccc6b873b
sensor: Uganda Demo 20260427222424 GEOGLOWS Forecast API | uuid=019dd10b-72e7-70f3-9ba4-f5f644300bce
processing_level: FORECAST_20260427222424 | uuid=019dd10b-764b-704e-9809-4008bc33286c
datastream: Uganda Demo 20260427222424 Aroca River GEOGLOWS Forecast | uuid=019dd10b-79d2-7d1e-a92a-cdbb665fdd67


,resource_type,station_id,source,name_or_code,uuid,python_type
0,thing,None,Aroca River,Uganda Demo 20260427222424 Aroca River GEOGLOW...,019dd10b-68b2-7e63-a68b-851252b222b7,Thing
1,observed_property,None,NaN,Uganda Demo 20260427222424 Forecast Streamflow,019dd10b-6c18-785f-98f9-fc7073faef17,ObservedProperty
2,unit,None,NaN,Uganda Demo 20260427222424 Cubic meters per se...,019dd10b-6f7f-74b8-a8a4-5d7ccc6b873b,Unit
3,sensor,None,NaN,Uganda Demo 20260427222424 GEOGLOWS Forecast API,019dd10b-72e7-70f3-9ba4-f5f644300bce,Sensor
4,processing_level,None,NaN,FORECAST_20260427222424,019dd10b-764b-704e-9809-4008bc33286c,ProcessingLevel
5,datastream,None,GEOGLOWS Forecast,Uganda Demo 20260427222424 Aroca River GEOGLOW...,019dd10b-79d2-7d1e-a92a-cdbb665fdd67,Datastream


## Configure and Optionally Run ETL

In [13]:
from hydroserverpy.etl import ETLPipeline
from hydroserverpy.etl.extractors import HTTPExtractor
from hydroserverpy.etl.loaders import HydroServerLoader
from hydroserverpy.etl.transformers import CSVTransformer, ETLDataMapping, ETLTargetPath

geoglows_forecast_source_uri = (
    f"{GEOGLOWS_API_ROOT.rstrip('/')}/ForecastStats/"
    f"?reach_id={AROCA_GEOGLOWS_RIVER_ID}&return_format=csv"
)
extractor = HTTPExtractor(source_uri=geoglows_forecast_source_uri)
transformer = CSVTransformer(timestamp_key="datetime", delimiter=",", header_row=1, data_start_row=2, timezone_type="utc")
forecast_target_id = AROCA_FORECAST_DATASTREAM_ID or resource_uid(forecast_datastream) or "<aroca-forecast-datastream-uuid>"
data_mappings = [ETLDataMapping(source_identifier="streamflow", target_paths=[ETLTargetPath(target_identifier=forecast_target_id)])]

print("GEOGLOWS forecast URL:")
print(geoglows_forecast_source_uri)
print(f"Aroca River ID: {AROCA_GEOGLOWS_RIVER_ID}; lat/lon: {AROCA_LATITUDE}, {AROCA_LONGITUDE}")

if not RUN_ETL:
    print("ETL run skipped because RUN_ETL is False.")
elif hs_api is None or not forecast_target_id:
    print("ETL run skipped because an authenticated client and target datastream are required.")
else:
    pipeline = ETLPipeline(extractor=extractor, transformer=transformer, loader=HydroServerLoader(client=hs_api, chunk_size=5000))
    context = pipeline.run(data_mappings=data_mappings, raise_on_error=False)
    print(context.status)
    print(context.stage)
    if context.results is not None:
        print(context.results.values_loaded_total)
        for target_id, target in context.results.target_results.items():
            print(target_id, target.status, getattr(target, "values_loaded", None))
    if getattr(context, "error", None):
        print(context.error)
        print(context.traceback)

GEOGLOWS forecast URL:
https://geoglows.ecmwf.int/api/ForecastStats/?reach_id=160180844&return_format=csv
Aroca River ID: 160180844; lat/lon: 1.533355, 32.21666
ETLStatus.FAILED
ETLStage.EXTRACT


## Cleanup: Delete Created Resources

In [ ]:
cleanup_order = [
    "task",
    "data_connection",
    "orchestration_system",
    "datastream",
    "thing",
    "result_qualifier",
    "processing_level",
    "sensor",
    "unit",
    "observed_property",
    "workspace",
]

preview = created_resources_dataframe()
if not preview.empty:
    display(preview)
else:
    print("No created resources are recorded.")

if not DELETE_CREATED_RESOURCES_AT_END:
    print("Cleanup skipped because DELETE_CREATED_RESOURCES_AT_END is False.")
elif hs_api is None:
    print("Cleanup skipped because the HydroServer client is unavailable.")
else:
    deleted_ids = set()
    for resource_type in cleanup_order:
        for row in reversed(created_resources):
            if row["resource_type"] != resource_type:
                continue
            resource = row["resource"]
            uid = resource_uid(resource)
            if resource is None or uid in deleted_ids:
                continue
            try:
                print(f"Deleting {resource_type}: {row['name_or_code']} | uuid={uid}")
                resource.delete()
                deleted_ids.add(uid)
            except Exception as exc:
                print(f"Could not delete {resource_type} {uid}: {exc}")
    print("Cleanup finished.")